## Overview

[Original data set: IBM HR Analytics Employee Attrition & Performance](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset/)

| Name                       | Description                                                                                                                                                    |
|----------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------|
| AGE                        | Numerical Value                                                                                                                                                |
| ATTRITION                  | Employee leaving the company (0=no, 1=yes)                                                                                                                     |
| BUSINESS TRAVEL            | (1=No Travel, 2=Travel Frequently, 3=Tavel Rarely)                                                                                                             |
| DAILY RATE                 | Numerical Value - Salary Level                                                                                                                                 |
| DEPARTMENT                 | (1=HR, 2=R&D, 3=Sales)                                                                                                                                         |
| DISTANCE FROM HOME         | Numerical Value - THE DISTANCE FROM WORK TO HOME                                                                                                               |
| EDUCATION                  | Numerical Value                                                                                                                                                |
| EDUCATION FIELD            | (1=HR, 2=LIFE SCIENCES, 3=MARKETING, 4=MEDICAL SCIENCES, 5=OTHERS, 6= TEHCNICAL)                                                                               |
| EMPLOYEE COUNT             | Numerical Value                                                                                                                                                |
| EMPLOYEE NUMBER            | Numerical Value - EMPLOYEE ID                                                                                                                                  |
| ENVIROMENT SATISFACTION    | Numerical Value - SATISFACTION WITH THE ENVIROMENT                                                                                                             |
| GENDER                     | (1=FEMALE, 2=MALE)                                                                                                                                             |
| HOURLY RATE                | Numerical Value - HOURLY SALARY                                                                                                                                |
| JOB INVOLVEMENT            | Numerical Value - JOB INVOLVEMENT                                                                                                                              |
| JOB LEVEL                  | Numerical Value - LEVEL OF JOB                                                                                                                                 |
| JOB ROLE                   | (1=HC REP, 2=HR, 3=LAB TECHNICIAN, 4=MANAGER, 5= MANAGING DIRECTOR, 6= REASEARCH DIRECTOR, 7= RESEARCH SCIENTIST, 8=SALES EXECUTIEVE, 9= SALES REPRESENTATIVE) |
| JOB SATISFACTION           | Numerical Value - SATISFACTION WITH THE JOB                                                                                                                    |
| MARITAL STATUS             | (1=DIVORCED, 2=MARRIED, 3=SINGLE)                                                                                                                              |
| MONTHLY INCOME             | Numerical Value - MONTHLY SALARY                                                                                                                               |
| MONTHY RATE                | Numerical Value - MONTHY RATE                                                                                                                                  |
| NUMCOMPANIES WORKED        | Numerical Value - NO. OF COMPANIES WORKED AT                                                                                                                   |
| OVER 18                    | (1=YES, 2=NO)                                                                                                                                                  |
| OVERTIME                   | (1=NO, 2=YES)                                                                                                                                                  |
| PERCENT SALARY HIKE        | Numerical Value - PERCENTAGE INCREASE IN SALARY                                                                                                                |
| PERFORMANCE RATING         | Numerical Value - ERFORMANCE RATING                                                                                                                            |
| RELATIONS SATISFACTION     | Numerical Value - RELATIONS SATISFACTION                                                                                                                       |
| STANDARD HOURS             | Numerical Value - STANDARD HOURS                                                                                                                               |
| STOCK OPTIONS LEVEL        | Numerical Value - STOCK OPTIONS                                                                                                                                |
| TOTAL WORKING YEARS        | Numerical Value - TOTAL YEARS WORKED                                                                                                                           |
| TRAINING TIMES LAST YEAR   | Numerical Value - HOURS SPENT TRAINING                                                                                                                         |
| WORK LIFE BALANCE          | Numerical Value - TIME SPENT BEWTWEEN WORK AND OUTSIDE                                                                                                         |
| YEARS AT COMPANY           | Numerical Value - TOTAL NUMBER OF YEARS AT THE COMPNAY                                                                                                         |
| YEARS IN CURRENT ROLE      | Numerical Value -YEARS IN CURRENT ROLE                                                                                                                         |
| YEARS SINCE LAST PROMOTION | Numerical Value - LAST PROMOTION                                                                                                                               |
| YEARS WITH CURRENT MANAGER | Numerical Value - YEARS SPENT WITH CURRENT MANAGER                                                                                                             |

In [ ]:
from pathlib import Path
import functools
import json

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, RepeatedKFold, RepeatedStratifiedKFold, GridSearchCV, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.base import BaseEstimator, TransformerMixin

from category_encoders import TargetEncoder, LeaveOneOutEncoder, WOEEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import torch
DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'

import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import seaborn as sns
sns.set_style('whitegrid')

# Uncomment to use AutoML
!pip install -q flaml
import flaml

!pip install -q autogluon.tabular[all]
from autogluon.tabular import TabularPredictor

## Data

In [ ]:
df_train = pd.read_csv('/kaggle/input/playground-series-s3e3/train.csv', index_col='id')
df_test = pd.read_csv('/kaggle/input/playground-series-s3e3/test.csv', index_col='id')
df_original = pd.read_csv('/kaggle/input/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv', index_col='EmployeeNumber')

print(df_train.info())
df_train.head()

In [ ]:
TARGET = 'Attrition'
print('Train:', len(df_train))
print(df_train[TARGET].value_counts())
print()
print('Test:', len(df_test))
print()
print('Original:', len(df_original))
print(df_original[TARGET].value_counts())

In [ ]:
df_original[TARGET] = (df_original[TARGET] == 'Yes').astype(int)
df_original[TARGET].value_counts()

## EDA
### Distribution of all variables

In [ ]:
ncols = 2
nrows = np.ceil(len(df_train.columns)/ncols).astype(int)
fig, axs = plt.subplots(ncols=ncols, nrows=nrows, figsize=(12,nrows*2.5))
for c, ax in zip(df_train.columns, axs.flatten()):
    sns.histplot(df_train, x=c, ax=ax)
fig.suptitle('Distribution of all variables', fontsize=20)
plt.tight_layout(rect=[0, 0, 1, 0.98])

### Count unique values

In [ ]:
counts_df = pd.concat([
    pd.DataFrame(df_train.drop(columns=[TARGET]).nunique(), columns=['train']),
    pd.DataFrame(df_test.nunique(), columns=['test']),
    pd.DataFrame(df_original.drop(columns=[TARGET]).nunique(), columns=['original']), 
    pd.DataFrame(df_train.drop(columns=[TARGET]).dtypes, columns=['dtype'])
], axis=1)
counts_df['top10values-train'] = counts_df.index.map(lambda d: df_train[d].value_counts().index[:10].tolist())
counts_df['top10values-test'] = counts_df.index.map(lambda d: df_test[d].value_counts().index[:10].tolist())
counts_df['count-difference'] = counts_df['train'] != counts_df['test']
counts_df.sort_values('train')

- `Over18`, `EmployeeCount`, `StandardHours` have a single value, carrying no information, drop them
- Based on the feature description and count of unique values, we can define categorical variables as the ones with count <= 9.
- Note that there are some variables with different counts between train and test sets.

In [ ]:
df_train.drop(columns=['Over18', 'EmployeeCount', 'StandardHours'], inplace=True, errors='ignore')
df_test.drop(columns=['Over18', 'EmployeeCount', 'StandardHours'], inplace=True, errors='ignore')
df_original.drop(columns=['Over18', 'EmployeeCount', 'StandardHours'], inplace=True, errors='ignore')
counts_df.drop(index=['Over18', 'EmployeeCount', 'StandardHours'], inplace=True, errors='ignore')

In [ ]:
NUM_FEATURES = counts_df[counts_df['train'] >= 10].index.tolist()
NUM_FEATURES

In [ ]:
CAT_FEATURES = sorted(set(counts_df.index).difference(NUM_FEATURES))
CAT_FEATURES

#### Outliers

In [ ]:
df_train.sort_values('DailyRate', ascending=False).head(3)[['DailyRate']]

In [ ]:
df_train.sort_values('Education', ascending=False).head(3)[['Education']]

### Compare distribution of variables grouped by the target variable

In [ ]:
TARGET_CAT = 'Attrition_cat'

In [ ]:
df_train[TARGET_CAT] = df_train[TARGET].astype('category')
df_original[TARGET_CAT] = df_original[TARGET].astype('category')
df_train['source'] = 'train'
df_original['source'] = 'original'
df_combined = pd.concat([df_train, df_original])

In [ ]:
fig, axs = plt.subplots(ncols=ncols, nrows=nrows, figsize=(12,nrows*3))
for c, ax in zip(df_train.columns, axs.flatten()):
    if c in NUM_FEATURES:
        sns.boxplot(data=df_train, x=c, y=TARGET_CAT, ax=ax)
    else:
        sns.countplot(data=df_train, x=c, hue=TARGET, ax=ax)
fig.suptitle('Distribution of variables grouped by the target variable', fontsize=20)
plt.tight_layout(rect=[0, 0, 1, 0.98])

### Compare distribution of variables between the generated and original datasets grouped by the target variable

In [ ]:
# fig, axs = plt.subplots(ncols=ncols, nrows=nrows, figsize=(12,nrows*3))
# for c, ax in zip(df_combined.columns, axs.flatten()):
#     if c in NUM_FEATURES:
#         sns.boxplot(data=df_combined, x=c, y=TARGET_CAT, hue='source', ax=ax)
#     else:
#         sns.barplot(data=df_combined, x=c, y=TARGET, hue='source', ax=ax)
# fig.suptitle('Distribution of variables grouped by the target variable and source', fontsize=20)
# plt.tight_layout(rect=[0, 0, 1, 0.98])

In [ ]:
df_train.drop(columns=['source', TARGET_CAT], inplace=True)
df_original.drop(columns=['source', TARGET_CAT], inplace=True)

### Missing values

In [ ]:
pd.concat([
    pd.DataFrame(df_train.drop(columns=[TARGET]).isnull().sum(), columns=['train']),
    pd.DataFrame(df_test.isnull().sum(), columns=['test']),
    pd.DataFrame(df_original.drop(columns=[TARGET]).isnull().sum(), columns=['original'])], axis=1)


## Categorical encoding

In [ ]:
y_train = df_train[TARGET]
df_train = df_train.drop(columns=[TARGET])
y_original = df_original[TARGET]
df_original = df_original.drop(columns=[TARGET])
df_train['original'] = 0
df_original['original'] = 1
df_test['original'] = 0
CAT_FEATURES += ['original']
DEFAULT_CV = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

In [ ]:
def build_pipeline(model_fn=None, num_attributes=None, cat_attributes=None, cat_encoder=None):
    num_proc = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler())
    cat_proc = make_pipeline(cat_encoder())
    processing = ColumnTransformer([
        ('num', num_proc, num_attributes or NUM_FEATURES),
        ('cat', cat_proc, cat_attributes or CAT_FEATURES)
    ])
    
    return Pipeline([ 
        ('proc', processing),
        ('model', model_fn())
    ])

def run(df_train, y_train, df_org=None, y_org=None, df_test=None, cv=DEFAULT_CV, pipeline_fn=None, use_lgbm_cat=False, return_models=False, save_file=None, verbose=False):
    oof = np.zeros(len(df_train))
    pipelines = []

    for fold, (idx_tr, idx_vl) in enumerate(cv.split(df_train, y_train)):
        # Fold train: add the entire original data
        df_tr, y_tr = df_train.iloc[idx_tr], y_train[idx_tr]
        if df_org is not None:
            df_tr = pd.concat([df_tr, df_org])
            y_tr = np.hstack([y_tr, y_org])
            
        # Fold validation: just provided data
        df_vl, y_vl = df_train.iloc[idx_vl], y_train[idx_vl]
         
        pipeline = pipeline_fn()
        
        if use_lgbm_cat:
            num_features_count = len(pipeline['proc'].transformers[0][2])
            cat_features_count = len(pipeline['proc'].transformers[1][2])
            pipeline.fit(df_tr, y_tr, model__categorical_feature=list(range(num_features_count, num_features_count+cat_features_count)))
        else:
            if type(pipeline['model']) == CatBoostClassifier:
                pipeline.fit(df_tr, y_tr, model__verbose=0)
            else:
                pipeline.fit(df_tr, y_tr)
        
        if hasattr(pipeline['model'], 'predict_proba'):
            oof[idx_vl] = pipeline.predict_proba(df_vl)[:,1]
        else:
            oof[idx_vl] = pipeline.predict(df_vl)
            
        score = roc_auc_score(y_vl, oof[idx_vl])
        pipelines.append(pipeline)
        
        if verbose:
            print(f'Fold {fold} rocauc={score:.4}')

    print(f'   OOF rocauc={roc_auc_score(y_train, oof):.4}')
    
    if save_file is not None:
        df = pd.DataFrame(data={'id': df_train.index, TARGET: oof})
        df.to_csv(f'oof_preds_{save_file}.csv', index=None)
        if df_test is not None:
            y_pred = np.mean([p.predict_proba(df_test)[:,1] for p in pipelines], axis=0)
            df = pd.DataFrame(data={'id': df_test.index, TARGET: y_pred})
            df.to_csv(f'test_preds_{save_file}.csv', index=None)
        
    
    if return_models:
        return pipelines

In [ ]:
def build_logreg(C=1): return LogisticRegression(C=C, max_iter=1000, random_state=0)
def encode_onehot(): return OneHotEncoder(handle_unknown='ignore')
def encode_ordinal(): return OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
def encode_target(): return TargetEncoder(min_samples_leaf=5, smoothing=5, cols=CAT_FEATURES)
def encode_loo(): return LeaveOneOutEncoder(cols=CAT_FEATURES, sigma=0.05)
def encode_woe(): return WOEEncoder(cols=CAT_FEATURES, sigma=0.05)

def compare_cat_encoding():
    print('Onehot')
    run(df_train, y_train, pipeline_fn=functools.partial(build_pipeline, model_fn=build_logreg, cat_encoder=encode_onehot))
    print('Ordinal')
    run(df_train, y_train, pipeline_fn=functools.partial(build_pipeline, model_fn=build_logreg, cat_encoder=encode_ordinal))
    print('Target')
    run(df_train, y_train, pipeline_fn=functools.partial(build_pipeline, model_fn=build_logreg, cat_encoder=encode_target))
    print('Leave One Out')
    run(df_train, y_train, pipeline_fn=functools.partial(build_pipeline, model_fn=build_logreg, cat_encoder=encode_loo))
    print('Weight of Evidence')
    run(df_train, y_train, pipeline_fn=functools.partial(build_pipeline, model_fn=build_logreg, cat_encoder=encode_woe))
    
compare_cat_encoding()

> One hot has the highest score. I'm not confident in using it because it generates quiet a lot of extra columns and we have very little data to train. WoE isn't too far back and might be a better choice?

In [ ]:
TUNED_CAT_ENCODER = encode_woe


## Tune logistic regression

In [ ]:
pipeline = build_pipeline(model_fn=build_logreg, cat_encoder=TUNED_CAT_ENCODER)
parameters = {'model__C': np.logspace(-2, 1, 20)}
search = GridSearchCV(pipeline, parameters, cv=DEFAULT_CV, scoring='roc_auc')
search.fit(df_train, y_train)
search.best_params_, search.best_score_

> A bit high C?

In [ ]:
df = pd.DataFrame({
    'feature': NUM_FEATURES + CAT_FEATURES,
    'coef': search.best_estimator_[1].coef_[0]
})
df.sort_values('coef').plot.barh(x='feature', figsize=(10,8))

In [ ]:
optimal_C = search.best_params_['model__C']
tuned_pipeline_logreg = functools.partial(build_pipeline, 
                                          model_fn=functools.partial(build_logreg, C=optimal_C), 
                                          cat_encoder=TUNED_CAT_ENCODER)
run(df_train, y_train, cv=DEFAULT_CV, pipeline_fn=tuned_pipeline_logreg)

## Add original data

### Use all

In [ ]:
run(df_train, y_train, df_org=df_original, y_org=y_original, cv=DEFAULT_CV, pipeline_fn=tuned_pipeline_logreg)

### Use only positive data

In [ ]:
y_original_pos = y_original[y_original == 1]
df_original_pos = df_original[y_original == 1]
assert y_original_pos.mean() == 1
assert len(y_original_pos) == len(df_original_pos)
run(df_train, y_train, df_org=df_original_pos, y_org=y_original_pos, cv=DEFAULT_CV, pipeline_fn=tuned_pipeline_logreg)

> Not quite good. Just include all.

## LightGBM

In [ ]:
def build_lgbm(): return LGBMClassifier(random_state=0)

pipeline_lgbm = functools.partial(build_pipeline, model_fn=build_lgbm, cat_encoder=TUNED_CAT_ENCODER)
run(df_train, y_train, df_original, y_original, cv=DEFAULT_CV, pipeline_fn=pipeline_lgbm)

### Try its own handling of categorical features

In [ ]:
pipeline_lgbm_cat = functools.partial(build_pipeline, model_fn=build_lgbm, cat_encoder=encode_ordinal)
run(df_train, y_train, df_original, y_original, cv=DEFAULT_CV, pipeline_fn=pipeline_lgbm_cat, use_lgbm_cat=True)

## AutoML

In [ ]:
df_combined = pd.concat([df_train, df_original])
y_combined = np.hstack([y_train, y_original.tolist()])
processor = pipeline_lgbm()['proc']
X_combined = processor.fit_transform(df_combined, y_combined)
df_combined_transformed = pd.DataFrame(X_combined, columns=processor.transformers[0][2] + processor.transformers[1][2])
df_combined_transformed[TARGET] = y_combined

X_train = processor.fit_transform(df_train, y_train)
df_train_transformed = pd.DataFrame(X_train, columns=processor.transformers[0][2] + processor.transformers[1][2])
df_train_transformed[TARGET] = y_train

In [ ]:
TIME_BUDGET = 60 * 60 * 2
EARLY_STOPPING_ROUNDS = 500
OUTPUT_FOLDER = Path('/kaggle/input/weight-of-evidence-automl/')

### FLAML

In [ ]:
# for model in ['lgbm', 'xgboost', 'catboost']:
#     auto_flaml = flaml.AutoML()
#     auto_flaml.fit(X_combined, y_combined, task='classification', metric='roc_auc', estimator_list=[model], time_budget=TIME_BUDGET, early_stop=EARLY_STOPPING_ROUNDS, verbose=0)
#     print(model)
#     print(auto_flaml.best_config)
#     print()
#     with open(f'tuned_{TIME_BUDGET}_{model}.json', 'w') as f:
#         f.write(json.dumps(auto_flaml.best_config))

### AutoGluon

In [ ]:
predictor = TabularPredictor(label=TARGET, problem_type='binary', eval_metric='roc_auc', path=f'AutoGluon_{TIME_BUDGET}')
predictor.fit(df_combined_transformed, time_limit=TIME_BUDGET, presets='best_quality', verbosity=0)
y_pred = predictor.get_oof_pred_proba(train_data=df_train_transformed).values[:,1]
df = pd.DataFrame(data={'id': df_combined_transformed.index, TARGET: y_pred})
df = df.iloc[:len(df_train)]
df.to_csv(f'oof_preds_woe_autogluon_both_data.csv', index=None)
        
X_test = processor.transform(df_test)
df_temp = pd.DataFrame(X_test, columns=processor.transformers[0][2] + processor.transformers[1][2])
y_pred = predictor.predict_proba(df_temp).values[:,1]
df = pd.DataFrame(data={'id': df_test.index, TARGET: y_pred})
df.to_csv(f'test_preds_woe_autogluon_both_data.csv', index=None)

## AutoML with just synthetic data

In [ ]:
# for model in ['lgbm', 'xgboost', 'catboost']:
#     auto_flaml = flaml.AutoML()
#     auto_flaml.fit(X_train, y_train, task='classification', metric='roc_auc', estimator_list=[model], time_budget=TIME_BUDGET, early_stop=EARLY_STOPPING_ROUNDS, verbose=0)
#     print(model)
#     print(auto_flaml.best_config)
#     print()
#     with open(f'synthetic_tuned_{TIME_BUDGET}_{model}.json', 'w') as f:
#         f.write(json.dumps(auto_flaml.best_config))

In [ ]:
predictor = TabularPredictor(label=TARGET, problem_type='binary', eval_metric='roc_auc', path=f'AutoGluon_synthetic_{TIME_BUDGET}')
predictor.fit(df_train_transformed, time_limit=TIME_BUDGET, presets='best_quality', verbosity=0)
y_pred = predictor.get_oof_pred_proba(train_data=df_train_transformed).values[:,1]
df = pd.DataFrame(data={'id': df_train.index, TARGET: y_pred})
df.to_csv(f'oof_preds_woe_autogluon_synthetic_data.csv', index=None)
        
X_test = processor.transform(df_test)
df_temp = pd.DataFrame(X_test, columns=processor.transformers[0][2] + processor.transformers[1][2])
y_pred = predictor.predict_proba(df_temp).values[:,1]
df = pd.DataFrame(data={'id': df_test.index, TARGET: y_pred})
df.to_csv(f'test_preds_woe_autogluon_synthetic_data.csv', index=None)

## Build an ensemble with hill climbing for both data

### Individual models

In [ ]:
# 1. Onehot logistic regression
pipeline = build_pipeline(model_fn=build_logreg, cat_encoder=encode_onehot)
parameters = {'model__C': np.logspace(-2, 1, 20)}
search = GridSearchCV(pipeline, parameters, cv=DEFAULT_CV, scoring='roc_auc')
search.fit(df_train, y_train)
optimal_C = search.best_params_['model__C']
tuned_pipeline_logreg = functools.partial(build_pipeline, 
                                          model_fn=functools.partial(build_logreg, C=optimal_C), 
                                          cat_encoder=encode_onehot)
run(df_train, y_train, df_org=df_original, y_org=y_original, df_test=df_test, cv=DEFAULT_CV, pipeline_fn=tuned_pipeline_logreg, save_file='onehot_logit_both_data')

# 2. WoE logistic regression
pipeline = build_pipeline(model_fn=build_logreg, cat_encoder=encode_woe)
search.fit(df_train, y_train)
optimal_C = search.best_params_['model__C']
tuned_pipeline_logreg = functools.partial(build_pipeline, 
                                          model_fn=functools.partial(build_logreg, C=optimal_C), 
                                          cat_encoder=encode_woe)
run(df_train, y_train, df_original, y_original, df_test, cv=DEFAULT_CV, pipeline_fn=tuned_pipeline_logreg, save_file='woe_logit_both_data')

In [ ]:
# Tuned params
cb_params = {
  "early_stopping_rounds": 10,
  "learning_rate": 0.06233639237958607,
  "n_estimators": 82
}

lgbm_params = {
  "n_estimators": 2126,
  "num_leaves": 4,
  "min_child_samples": 4,
  "learning_rate": 0.07481169681846314,
  "log_max_bin": 5,
  "colsample_bytree": 0.2907389188173741,
  "reg_alpha": 0.08703138712948819,
  "reg_lambda": 261.89784654488506
}

xgb_params = {
  "n_estimators": 642,
  "max_leaves": 4,
  "min_child_weight": 10.071459921844403,
  "learning_rate": 0.03170611830917708,
  "subsample": 0.9658333237522234,
  "colsample_bylevel": 0.34024643285914785,
  "colsample_bytree": 0.6284060610772222,
  "reg_alpha": 0.011822289129143907,
  "reg_lambda": 0.0009765625
}

for model in ['lgbm', 'xgboost', 'catboost']:
    print(model)

    def build_model(): 
        if model == 'lgbm': return LGBMClassifier(**lgbm_params, random_state=0)
        if model == 'xgboost': return XGBClassifier(**xgb_params, random_state=0)
        if model == 'catboost': return CatBoostClassifier(**cb_params, random_state=0)

    pipeline_fn = functools.partial(build_pipeline, model_fn=build_model, cat_encoder=TUNED_CAT_ENCODER)
    run(df_train, y_train, df_original, y_original, df_test, cv=DEFAULT_CV, pipeline_fn=pipeline_fn, save_file=f'woe_{model}_both_data')

### Hill climbing
Source: https://www.kaggle.com/code/samuelcortinhas/ps-s3e3-hill-climbing-like-a-gm/notebook

In [ ]:
def join_preds(models):
    # Join oof preds
    oof_df = pd.DataFrame(index=np.arange(len(y_train)))
    for m in models:
        df = pd.read_csv(f'oof_preds_{m}.csv').drop(columns=['id'])
        df.rename(columns={"Attrition": m}, inplace=True)
        oof_df = pd.concat([oof_df,df], axis=1)

    # Join test preds
    test_preds = pd.DataFrame(index=np.arange(len(df_test)))
    for m in models:
        df = pd.read_csv(f'test_preds_{m}.csv').drop(columns=['id'])
        df.rename(columns={"Attrition": m}, inplace=True)
        test_preds = pd.concat([test_preds,df], axis=1)
        
    # Evaluate oof preds
    scores = {}
    for col in oof_df.columns:
        scores[col] = roc_auc_score(y_train, oof_df[col])

    # Sort scores
    scores = {k: v for k, v in sorted(scores.items(), key=lambda item: item[1], reverse=True)}

    # Sort oof_df and test_preds
    oof_df = oof_df[list(scores.keys())]
    test_preds = test_preds[list(scores.keys())]

    return oof_df, test_preds

def climb(oof_df, test_preds, y):
    # Initialise
    STOP = False
    current_best_ensemble = oof_df.iloc[:,0]
    current_best_test_preds = test_preds.iloc[:,0]
    MODELS = oof_df.iloc[:,1:]
    weight_range = np.arange(0.01,0.51,0.01)   # or with negative weights: np.arange(-0.5,0.51,0.01)
    history = [roc_auc_score(y, current_best_ensemble)]
    i=0

    # Hill climbing
    while not STOP:
        i+=1
        potential_new_best_cv_score = roc_auc_score(y, current_best_ensemble)
        k_best, wgt_best = None, None
        for k in MODELS:
            for wgt in weight_range:
                potential_ensemble = (1-wgt) * current_best_ensemble + wgt * MODELS[k]
                cv_score = roc_auc_score(y, potential_ensemble)
                if cv_score > potential_new_best_cv_score:
                    potential_new_best_cv_score = cv_score
                    k_best, wgt_best = k, wgt

        if k_best is not None:
            current_best_ensemble = (1-wgt_best) * current_best_ensemble + wgt_best * MODELS[k_best]
            current_best_test_preds = (1-wgt_best) * current_best_test_preds + wgt_best * test_preds[k_best]
            MODELS = MODELS.drop(k_best, axis=1)
            if MODELS.shape[1]==0:
                STOP = True
            print(f'Iteration: {i}, Model added: {k_best}, Best weight: {wgt_best:.2f}, Best AUC: {potential_new_best_cv_score:.5f}')
            history.append(potential_new_best_cv_score)
        else:
            STOP = True
            
    plt.figure(figsize=(10,4))
    plt.plot(np.arange(len(history))+1, history, marker="x")
    plt.title("CV AUC vs. Number of Models with Hill Climbing")
    plt.xlabel("Number of models")
    plt.ylabel("AUC")
    plt.show()
    
    return current_best_test_preds

In [ ]:
models = ['onehot_logit', 'woe_logit', 'woe_lgbm', 'woe_catboost', 'woe_xgboost', 'woe_autogluon']
models = [m + '_both_data' for m in models]

oof_df, test_preds = join_preds(models)
current_best_test_preds = climb(oof_df, test_preds, y_train)
submission = pd.DataFrame(data={'id': df_test.index, TARGET: current_best_test_preds.values})
submission.to_csv('hill_both_data.csv', index=None)

## Build an ensemble with hill climbing for only synthetic data
### Individual models

In [ ]:
# 1. Onehot logistic regression
pipeline = build_pipeline(model_fn=build_logreg, cat_encoder=encode_onehot)
parameters = {'model__C': np.logspace(-2, 1, 20)}
search = GridSearchCV(pipeline, parameters, cv=DEFAULT_CV, scoring='roc_auc')
search.fit(df_train, y_train)
optimal_C = search.best_params_['model__C']
tuned_pipeline_logreg = functools.partial(build_pipeline, 
                                          model_fn=functools.partial(build_logreg, C=optimal_C), 
                                          cat_encoder=encode_onehot)
run(df_train, y_train, df_test=df_test, cv=DEFAULT_CV, pipeline_fn=tuned_pipeline_logreg, save_file='onehot_logit_synthetic_data')

# 2. WoE logistic regression
pipeline = build_pipeline(model_fn=build_logreg, cat_encoder=encode_woe)
search.fit(df_train, y_train)
optimal_C = search.best_params_['model__C']
tuned_pipeline_logreg = functools.partial(build_pipeline, 
                                          model_fn=functools.partial(build_logreg, C=optimal_C), 
                                          cat_encoder=encode_woe)
run(df_train, y_train, df_test=df_test, cv=DEFAULT_CV, pipeline_fn=tuned_pipeline_logreg, save_file='woe_logit_synthetic_data')

# 3. Tree-based with AutoML
for model in ['lgbm', 'xgboost', 'catboost']:
    print(model)

    def build_model(): 
        if model == 'lgbm': return LGBMClassifier(**lgbm_params, random_state=0)
        if model == 'xgboost': return XGBClassifier(**xgb_params, random_state=0)
        if model == 'catboost': return CatBoostClassifier(**cb_params, random_state=0)

    pipeline_fn = functools.partial(build_pipeline, model_fn=build_model, cat_encoder=TUNED_CAT_ENCODER)
    run(df_train, y_train, df_test=df_test, cv=DEFAULT_CV, pipeline_fn=pipeline_fn, save_file=f'woe_{model}_synthetic_data')

In [ ]:
models = ['onehot_logit', 'woe_logit', 'woe_lgbm', 'woe_catboost', 'woe_xgboost', 'woe_autogluon']
models = [m + '_synthetic_data' for m in models]

oof_df, test_preds = join_preds(models)
current_best_test_preds = climb(oof_df, test_preds, y_train)
submission = pd.DataFrame(data={'id': df_test.index, TARGET: current_best_test_preds.values})
submission.to_csv('hill_synthetic_data.csv', index=None)